In [1]:
# Day 11: Prepare Data for Power BI Dashboard
# =============================================
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('../.env')
warehouse_connection = create_engine(os.getenv('DATABASE_URL'))
os.makedirs('../dashboard/data', exist_ok=True)

# ===== EXPORT 1: FUNNEL METRICS =====
funnel_data = pd.read_sql("""
    WITH user_stages AS (
        SELECT user_id,
            MAX(CASE WHEN event_type = 'page_view' THEN 1 ELSE 0 END) AS viewed_page,
            MAX(CASE WHEN event_type = 'product_view' THEN 1 ELSE 0 END) AS viewed_product,
            MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS added_cart,
            MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased
        FROM fact_events GROUP BY user_id
    )
    SELECT 'Page View' AS stage, 1 AS stage_order, SUM(viewed_page) AS users FROM user_stages
    UNION ALL SELECT 'Product View', 2, SUM(viewed_product) FROM user_stages
    UNION ALL SELECT 'Add to Cart', 3, SUM(added_cart) FROM user_stages
    UNION ALL SELECT 'Purchase', 4, SUM(purchased) FROM user_stages
    ORDER BY stage_order;
""", warehouse_connection)
funnel_data.to_csv('../dashboard/data/funnel_stages.csv', index=False)
print(f"Funnel stages: {len(funnel_data)} rows")

# ===== EXPORT 2: DAILY REVENUE =====
daily_revenue = pd.read_sql("""
    SELECT DATE(timestamp) AS event_date,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(*) AS total_events,
        SUM(CASE WHEN event_type = 'purchase' THEN amount ELSE 0 END) AS revenue,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS purchases
    FROM fact_events
    GROUP BY DATE(timestamp)
    ORDER BY event_date;
""", warehouse_connection)
daily_revenue.to_csv('../dashboard/data/daily_metrics.csv', index=False)
print(f"Daily metrics: {len(daily_revenue)} rows")

# ===== EXPORT 3: HOURLY PATTERN =====
hourly_pattern = pd.read_sql("""
    SELECT EXTRACT(HOUR FROM timestamp) AS hour_of_day,
        TO_CHAR(timestamp, 'Day') AS day_name,
        COUNT(*) AS event_count,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS purchases
    FROM fact_events
    GROUP BY EXTRACT(HOUR FROM timestamp), TO_CHAR(timestamp, 'Day')
    ORDER BY hour_of_day;
""", warehouse_connection)
hourly_pattern.to_csv('../dashboard/data/hourly_heatmap.csv', index=False)
print(f"Hourly heatmap: {len(hourly_pattern)} rows")

# ===== EXPORT 4: TOP PRODUCTS =====
top_products = pd.read_sql("""
    SELECT product_id,
        COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) AS purchases,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN amount ELSE 0 END)::numeric, 2) AS revenue
    FROM fact_events
    WHERE product_id IS NOT NULL
    GROUP BY product_id
    ORDER BY revenue DESC NULLS LAST
    LIMIT 20;
""", warehouse_connection)
top_products.to_csv('../dashboard/data/top_products.csv', index=False)
print(f"Top products: {len(top_products)} rows")

# ===== EXPORT 5: RFM SEGMENTS =====
rfm_data = pd.read_csv('../data/processed/rfm_segmentation.csv')
segment_summary = rfm_data.groupby('customer_segment').agg(
    shopper_count=('shopper_id', 'count'),
    total_revenue=('monetary', 'sum'),
    avg_revenue=('monetary', 'mean'),
    avg_recency=('recency_days', 'mean')
).round(2).reset_index()
segment_summary.to_csv('../dashboard/data/rfm_segments.csv', index=False)
print(f"RFM segments: {len(segment_summary)} rows")

# ===== EXPORT 6: KPI CARDS =====
kpis = pd.read_sql("""
    SELECT 
        (SELECT COUNT(DISTINCT user_id) FROM fact_events) AS total_users,
        (SELECT COUNT(*) FROM fact_events WHERE event_type = 'purchase') AS total_purchases,
        (SELECT ROUND(SUM(amount)::numeric, 2) FROM fact_events WHERE event_type = 'purchase') AS total_revenue,
        (SELECT ROUND(AVG(amount)::numeric, 2) FROM fact_events WHERE event_type = 'purchase') AS avg_order_value,
        (SELECT COUNT(DISTINCT session_id) FROM fact_events) AS total_sessions,
        (SELECT COUNT(DISTINCT product_id) FROM fact_events WHERE product_id IS NOT NULL) AS total_products
""", warehouse_connection)
kpis.to_csv('../dashboard/data/kpis.csv', index=False)
print(f"KPIs exported")

print("\n✅ All dashboard data files saved to dashboard/data/")
print("Files ready for Power BI import!")

Funnel stages: 4 rows
Daily metrics: 365 rows
Hourly heatmap: 168 rows
Top products: 20 rows
RFM segments: 6 rows
KPIs exported

✅ All dashboard data files saved to dashboard/data/
Files ready for Power BI import!
